[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeffvan302/unsloth-embedding-colab/blob/main/embedding_finetune_poc.ipynb)

# Embedding fine-tuning proof of concept (Unsloth + Qwen3-Embedding)

This notebook walks the whole loop on a **free Colab T4**:

1. **Load documents** (a built-in sample corpus, or your own files)
2. **Build a synthetic training library** — a small local LLM writes realistic questions for every chunk, and your glossary/word set becomes extra pairs
3. **Mine hard negatives** with the base model
4. **Measure the base model** on held-out documents (Recall@k, MRR, NDCG)
5. **Fine-tune** `Qwen3-Embedding-0.6B` with Unsloth + LoRA
6. **Measure again**, try the `search()` hook, and save the model

Every place you are meant to plug in your own material is marked **🔌 HOOK**.

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. End to end it takes roughly 10–15 minutes with the sample data.

> The sample corpus describes a *fictional* warehouse-robotics product ("Halyard") full of made-up jargon, so the base model cannot already know it — which is exactly the situation with your own internal docs.

## 1. Install

In [ ]:
%%capture
# Same install recipe as Unsloth's official Qwen3-Embedding notebook.
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.35")
    if str(torch.version.cuda).startswith("13") and xformers.endswith("0.0.35"): xformers = "https://download.pytorch.org/whl/cu130/xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl"
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2
!pip install pypdf

In [ ]:
# Import Unsloth first so its patches are applied before transformers is used.
import unsloth
from unsloth import FastSentenceTransformer, is_bf16_supported
import torch, transformers, sentence_transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| sentence-transformers", sentence_transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch the runtime to a T4 GPU")

## 2. 🔌 HOOK — Configuration

Everything you would change for your own run lives here.

In [ ]:
# ---- Models -------------------------------------------------------------
BASE_MODEL      = "unsloth/Qwen3-Embedding-0.6B"   # later: "Qwen/Qwen3-Embedding-8B" on the 2x96GB box
GENERATOR_MODEL = "Qwen/Qwen2.5-3B-Instruct"       # small local LLM that writes synthetic questions (fits a T4)

# ---- Your documents -------------------------------------------------------
# "sample"  -> the built-in fictional Halyard docs below
# "upload"  -> pick .md / .txt / .pdf files from your computer when the cell runs
# a folder  -> e.g. "/content/drive/MyDrive/my_docs" (run the Drive-mount cell first)
DOCS_SOURCE     = "sample"
GLOSSARY_CSV    = None        # optional path to a CSV with columns: term,definition

# ---- Synthetic data ---------------------------------------------------------
CHUNK_MAX_WORDS    = 80       # chunk size in words; paragraphs are packed up to this size
QUERIES_PER_CHUNK  = 5
TEST_FRACTION      = 0.2      # share of chunks held out entirely for evaluation
PAIRS_CACHE        = "synthetic_pairs.jsonl"   # generated library is saved here; delete it to regenerate
SEED               = 42

# Qwen3-Embedding expects an instruction on queries (documents get none).
TASK_INSTRUCTION = "Given a question about our internal documentation, retrieve the passage that answers it"
QUERY_PROMPT     = f"Instruct: {TASK_INSTRUCTION}\nQuery: "

# ---- Training -------------------------------------------------------------
MAX_SEQ_LEN    = 384
LORA_RANK      = 32
EPOCHS         = 4
BATCH_SIZE     = 32           # in-batch negatives: bigger is better if memory allows
LEARNING_RATE  = 5e-5
OUTPUT_DIR     = "halyard_embedding_lora"
HF_REPO        = None         # e.g. "your-username/my-embedding-model" to push to the Hub

## 3. Sample corpus and word set

A fictional product with its own vocabulary. Replace it by changing `DOCS_SOURCE`, or edit `GLOSSARY` to try your own terms.

In [ ]:
SAMPLE_DOCS = {
"fleet_overview.md": """Halyard runs two robot models on the warehouse floor. The Rook (model R-4) is a low, wheeled picking robot that carries totes between storage aisles and the packing stations. The Heron (model H-2) is a vertical lift robot that raises and lowers pallets in the high-bay racking. Both are coordinated by the Kestrel console, the browser-based control center used by shift supervisors.

A typical site runs 40 to 120 Rooks and 4 to 12 Herons. Rooks do the high-volume, small-item work; Herons only handle full pallets and never travel outside the high-bay zone. If a task needs both, Kestrel splits it into a Heron lift followed by a Rook tote handoff.""",

"dwell_lock.md": """A dwell lock is a safety state in which a robot stops and holds its exact position because two of its sensors disagree, for example the floor camera sees an obstacle that the lidar does not. While in dwell lock the robot keeps its brakes engaged and flashes amber. It will not move again on its own, even after the obstruction is gone.

To clear a dwell lock, a technician must physically walk to the robot, confirm the path is safe, and press the blue acknowledge button on its rear panel. Clearing it remotely from Kestrel is deliberately disabled. If the same robot enters dwell lock more than three times in one shift, Kestrel flags it for a sensor calibration before it can return to service.""",

"lantern_codes.md": """Lantern codes are the error codes reported by Heron lift robots. Every lantern code starts with L followed by three digits. Codes L-100 to L-199 are hydraulic faults, such as low fluid pressure or a slow mast extension. Codes L-200 to L-299 are load faults, meaning the pallet is overweight, off-center, or its weight changed during the lift.

Codes L-300 to L-399 cover positioning faults, where the Heron cannot confirm it is aligned with the rack beam. Codes L-400 and above are communication faults between the Heron and Kestrel. Any L-2xx load fault requires a human to inspect the pallet before the lift is retried; hydraulic and positioning faults can be retried once automatically.""",

"brine_cycle.md": """The brine cycle is the nightly battery recalibration that every Rook goes through. During the brine cycle the robot fully discharges to a safe floor level and then recharges slowly, which lets the battery management system re-learn the true capacity of the pack. It takes about 90 minutes and is scheduled automatically during the quietest hours of the night.

Skipping the brine cycle for more than five nights makes the reported battery percentage drift away from reality, and Rooks may then shut down mid-task with a charge reading of 20 percent or more. Supervisors can postpone a brine cycle from Kestrel, but never cancel it outright.""",

"cradle_swap.md": """A cradle swap is the procedure for replacing a Rook's battery cradle, the removable tray that holds its battery pack. Cradle swaps are needed when a battery is swollen, when the cradle latch is damaged, or when the pack has passed 1,500 charge cycles.

Before a cradle swap, put the Rook in burrow mode and wait for the status light to turn solid blue. Release the two side latches, slide the cradle out, and insert the replacement until both latches click. After the swap, the Rook must complete one full brine cycle before Kestrel will assign it work again.""",

"ghost_slots.md": """A ghost slot is a storage location that the inventory system believes is occupied but which is physically empty when a Rook arrives to pick from it. Ghost slots usually come from a missed scan during putaway or from an item that was moved by hand without being recorded.

When a Rook detects a ghost slot it reports the location to Kestrel, skips the pick, and the order is re-routed to another slot with the same item if one exists. Ghost slots are listed on the Kestrel exceptions page, and a stock controller should physically verify each one within 24 hours, because left alone they cause repeated failed picks.""",

"tally_drift.md": """Tally drift is the gradual difference between the stock count that Halyard records and the stock that is really on the shelves. Small amounts of tally drift are normal and come from damaged items, mis-scans, and ghost slots. Halyard reports drift per aisle as a percentage.

When tally drift in an aisle exceeds 2 percent, Kestrel schedules a cycle count for that aisle during the next quiet window. A cycle count sends a Rook with a camera mast down the aisle to re-scan every location. Drift above 5 percent is escalated to the site's inventory manager because it usually points to a process problem rather than random errors.""",

"pallet_echo.md": """A pallet echo happens when the same pallet ID is registered twice, usually because a pallet label was scanned at the dock and scanned again after being re-wrapped. Kestrel then believes there are two pallets in two different locations, and a Heron may be sent to lift a pallet that does not exist.

Kestrel automatically detects most pallet echoes by checking for identical IDs with arrival times less than one hour apart. Suspected echoes appear in amber on the high-bay map. To resolve one, a supervisor confirms which location really holds the pallet and retires the other record. Never re-label a pallet to fix an echo, as that hides the problem from the audit log.""",

"tote_handoff.md": """A tote handoff, written THO in logs, is the moment a Rook passes a tote onto a conveyor spur lane or to another robot. Spur lanes are the short conveyor branches that lead from the main conveyor loop to each packing station. A successful handoff is confirmed when the spur lane's photo-eye sees the tote and the Rook's load sensor reads empty.

Failed tote handoffs are almost always caused by a spur lane that is full or stopped. The Rook will wait at the spur for up to 60 seconds, then return the tote to a buffer location and report a THO timeout. Frequent THO timeouts at one station usually mean the packers there need more staff, not that the robots are faulty.""",

"quiet_window.md": """A quiet window is a scheduled period during which no software, firmware, or configuration changes may be applied to the Halyard fleet. Quiet windows are set around the busiest trading periods, such as the weeks before major holidays, and are configured by the site operations manager in Kestrel.

During a quiet window, firmware rollouts are paused automatically and configuration edits in Kestrel are read-only for everyone except users with the Site Owner role. Emergency fixes can still be applied, but they require a written approval from two Site Owners and are recorded in the change log with the reason.""",

"burrow_mode.md": """Burrow mode is a low-power parking state for Rooks. A Rook in burrow mode drives to the nearest parking bay, lowers its chassis onto the floor, switches off its drive motors and sensors except for a single heartbeat radio, and shows a solid blue light. Burrow mode is used overnight at sites with low volume, and before any maintenance work such as a cradle swap.

A Rook can be woken from burrow mode from Kestrel in about 30 seconds. Robots that stay in burrow mode for more than 14 days must be fully charged and run a brine cycle before they rejoin the fleet, because their battery calibration will be stale.""",

"wake_tokens.md": """A wake token is the one-time credential a new or repaired robot uses to join the fleet. Wake tokens are generated in Kestrel under Fleet, Add Robot, and are valid for 30 minutes. The technician enters the token on the robot's service screen, and the robot then downloads its site map, firmware, and safety zones.

Each wake token can be used only once and only for the robot model it was generated for, so a Heron token will be rejected by a Rook. If a token expires before the robot connects, simply generate a new one; expired tokens cannot be extended. Tokens are logged with the name of the user who created them.""",

"kestrel_roles.md": """Kestrel has four user roles. Viewers can see the live floor map and reports but cannot change anything. Supervisors can assign and pause work, clear exceptions such as ghost slots and pallet echoes, and postpone brine cycles. Technicians can put robots in burrow mode, generate wake tokens, and record maintenance such as cradle swaps.

Site Owners have every permission, including editing quiet windows, approving emergency changes, and managing other users. Roles are assigned per site, so a person can be a Site Owner at one warehouse and only a Viewer at another. Accounts that are not used for 90 days are automatically downgraded to Viewer.""",

"firmware_rings.md": """Halyard firmware is rolled out in rings so that a bad update never reaches the whole fleet at once. Ring 0 is a single designated test robot per site, ring 1 is ten percent of the fleet, and ring 2 is everyone else. Each ring must run for at least 48 hours without a regression before the next ring starts.

A regression is any rise in dwell locks, THO timeouts, or lantern codes compared with the previous week. If Kestrel detects a regression, the rollout halts and affected robots roll back to the last good firmware automatically. Rollouts never start or continue during a quiet window.""",

"escalation.md": """Halyard incidents are classified into three tiers. Tier 3 incidents affect a single robot and are handled by the on-site technician, for example a repeated dwell lock or a failed cradle swap. Tier 2 incidents affect a zone or a group of robots, such as all Herons in one high-bay aisle reporting the same lantern code, and must be reported to the regional support desk within one hour.

Tier 1 incidents stop or endanger the whole site, including any injury, a fire alarm, or the entire fleet losing contact with Kestrel. For tier 1, press the site emergency stop first, then call the 24-hour Halyard support line. Do not attempt to restart the fleet until support has confirmed it is safe.""",

"cold_storage.md": """Rooks can operate in chilled areas down to 0 degrees Celsius without changes. For freezer zones, down to minus 25 degrees Celsius, a site must use the cold-rated Rook variant, which has heated battery cradles and sealed wheel bearings. Standard Rooks that enter a freezer zone will be stopped by Kestrel at the zone boundary.

Cold-rated Rooks may spend at most 45 minutes inside a freezer zone before returning to a normal-temperature buffer area for 15 minutes to warm their batteries. Condensation can form on sensors when a robot leaves the freezer, so cold-rated Rooks automatically pause for two minutes at the exit to avoid false dwell locks.""",
}

# 🔌 HOOK — your word set. Term -> definition. Add, remove, or load from GLOSSARY_CSV.
GLOSSARY = {
    "dwell lock": "safety state where a robot freezes in place because two sensors disagree; cleared only by pressing the acknowledge button on the robot",
    "lantern code": "error code from a Heron lift robot, written L followed by three digits",
    "brine cycle": "nightly full discharge and slow recharge that recalibrates a Rook's battery",
    "cradle swap": "procedure for replacing the removable tray that holds a Rook's battery",
    "ghost slot": "storage location recorded as full that is physically empty",
    "tally drift": "difference between recorded stock and actual stock on the shelves",
    "pallet echo": "the same pallet ID registered twice, creating a phantom pallet",
    "tote handoff": "a Rook passing a tote onto a conveyor spur lane; abbreviated THO",
    "THO timeout": "a failed tote handoff because the spur lane was full or stopped",
    "spur lane": "short conveyor branch leading to a packing station",
    "quiet window": "change-freeze period when no firmware or configuration changes are allowed",
    "burrow mode": "low-power parking state for a Rook, shown by a solid blue light",
    "wake token": "one-time 30-minute credential a robot uses to join the fleet",
    "Kestrel": "the browser-based console used to control and monitor the Halyard fleet",
    "Rook": "Halyard's wheeled tote-picking robot, model R-4",
    "Heron": "Halyard's vertical pallet-lift robot, model H-2",
    "firmware ring": "stage of a firmware rollout: ring 0 test robot, ring 1 ten percent, ring 2 everyone",
}
print(len(SAMPLE_DOCS), "sample documents,", len(GLOSSARY), "glossary terms")

## 4. 🔌 HOOK — Load and chunk documents

`load_documents()` returns `{filename: text}`. Swap in your own loader (Confluence export, a database, a SharePoint sync…) as long as it returns that shape.

In [ ]:
# Optional: uncomment to read docs from Google Drive, then set DOCS_SOURCE to a Drive folder.
# from google.colab import drive; drive.mount("/content/drive")

In [ ]:
import os, re, glob, json, random, csv

def read_file(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf":
        from pypdf import PdfReader
        return "\n\n".join((p.extract_text() or "") for p in PdfReader(path).pages)
    with open(path, encoding="utf-8", errors="ignore") as f:
        return f.read()

def load_documents(source=DOCS_SOURCE):
    if source == "sample":
        return dict(SAMPLE_DOCS)
    if source == "upload":
        from google.colab import files
        uploaded = files.upload()
        return {name: read_file(name) for name in uploaded}
    paths = [p for ext in ("md", "txt", "pdf") for p in glob.glob(os.path.join(source, "**", f"*.{ext}"), recursive=True)]
    return {os.path.relpath(p, source): read_file(p) for p in sorted(paths)}

def chunk_text(text, max_words=CHUNK_MAX_WORDS):
    """Pack paragraphs into chunks of up to max_words; split overly long paragraphs on sentences."""
    paras = [re.sub(r"\s+", " ", p).strip() for p in re.split(r"\n\s*\n", text)]
    pieces = []
    for p in filter(None, paras):
        if len(p.split()) <= max_words:
            pieces.append(p)
            continue
        cur = []
        for s in re.split(r"(?<=[.!?])\s+", p):
            if cur and len(" ".join(cur + [s]).split()) > max_words:
                pieces.append(" ".join(cur)); cur = []
            cur.append(s)
        if cur: pieces.append(" ".join(cur))
    chunks, cur = [], ""
    for piece in pieces:
        if cur and len((cur + " " + piece).split()) > max_words:
            chunks.append(cur); cur = piece
        else:
            cur = (cur + "\n\n" + piece).strip()
    if cur: chunks.append(cur)
    return chunks

def load_glossary():
    glossary = dict(GLOSSARY) if DOCS_SOURCE == "sample" else {}
    if GLOSSARY_CSV:
        with open(GLOSSARY_CSV, encoding="utf-8") as f:
            for row in csv.DictReader(f):
                glossary[row["term"].strip()] = row["definition"].strip()
    return glossary

docs = load_documents()
chunks = []   # list of {"id", "doc", "text"}
for name, text in docs.items():
    for i, c in enumerate(chunk_text(text)):
        chunks.append({"id": f"{name}#{i}", "doc": name, "text": c})
glossary = load_glossary()
print(f"{len(docs)} documents -> {len(chunks)} chunks; {len(glossary)} glossary terms")
print("\nExample chunk:\n", chunks[1]["text"][:400])

## 5. 🔌 HOOK — Build the synthetic training library

A small instruct model reads each chunk and writes questions a real user might type. The result is saved to `synthetic_pairs.jsonl` — open it from the Files panel, review or hand-edit it, and re-run; the cache is reused until you delete it.

To use a stronger generator (e.g. a hosted LLM API), replace `generate_queries()` — it just has to take a list of chunk texts and return a list of question lists.

In [ ]:
import gc

GEN_SYSTEM = ("You write realistic search queries that operators, technicians and support staff "
              "type when looking for information in internal documentation.")
GEN_USER = """Here is a passage from our documentation:

\"\"\"{chunk}\"\"\"

Write {n} different questions or search queries that this passage answers.
Vary them: some short keyword-style queries, some full questions, and some that describe a symptom or situation instead of naming the term.
Do not copy long phrases from the passage. Output one per line with no numbering and nothing else."""

def clean_lines(raw, n):
    out = []
    for line in raw.splitlines():
        line = re.sub(r"^\s*(?:[-*•]|\d+[.)])\s*", "", line).strip().strip('"').strip()
        if 8 <= len(line) <= 200 and line.lower() not in {o.lower() for o in out}:
            out.append(line)
    return out[:n]

def generate_queries(chunk_texts, n=QUERIES_PER_CHUNK, batch_size=8):
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(GENERATOR_MODEL, padding_side="left")
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    llm = AutoModelForCausalLM.from_pretrained(GENERATOR_MODEL, torch_dtype=torch.float16, device_map="cuda")
    prompts = [tok.apply_chat_template(
                   [{"role": "system", "content": GEN_SYSTEM},
                    {"role": "user", "content": GEN_USER.format(chunk=c, n=n)}],
                   tokenize=False, add_generation_prompt=True) for c in chunk_texts]
    results = []
    for i in range(0, len(prompts), batch_size):
        enc = tok(prompts[i:i + batch_size], return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = llm.generate(**enc, max_new_tokens=60 * n, do_sample=True, temperature=0.7,
                               top_p=0.9, pad_token_id=tok.pad_token_id)
        texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        results += [clean_lines(t, n) for t in texts]
        print(f"  generated {min(i + batch_size, len(prompts))}/{len(prompts)} chunks")
    del llm; gc.collect(); torch.cuda.empty_cache()   # free the T4 before training
    return results

def glossary_pairs(glossary):
    """Turn the word set into (query, passage) pairs with simple templates — no LLM needed."""
    pairs = []
    for term, definition in glossary.items():
        for q in (f"What does {term} mean?", term, f"define {term}"):
            pairs.append({"query": q, "chunk_id": f"glossary::{term}", "source": "glossary"})
    return pairs

if os.path.exists(PAIRS_CACHE):
    with open(PAIRS_CACHE) as f:
        generated = [json.loads(l) for l in f]
    print(f"Loaded {len(generated)} cached synthetic queries from {PAIRS_CACHE}")
else:
    qs = generate_queries([c["text"] for c in chunks])
    generated = [{"query": q, "chunk_id": c["id"], "source": "llm"} for c, ql in zip(chunks, qs) for q in ql]
    with open(PAIRS_CACHE, "w") as f:
        for row in generated: f.write(json.dumps(row) + "\n")
    print(f"Saved {len(generated)} synthetic queries to {PAIRS_CACHE}")

glossary_rows = glossary_pairs(glossary)
chunk_by_id = {c["id"]: c["text"] for c in chunks}
chunk_by_id.update({f"glossary::{t}": f"{t}: {d}" for t, d in glossary.items()})

for row in random.Random(SEED).sample(generated, min(6, len(generated))):
    print(f"\nQ: {row['query']}\n   -> {row['chunk_id']}")

## 6. Train / test split

Whole **chunks** are held out, so evaluation questions point at passages the model never saw as a positive during training. The search corpus at evaluation time is *every* chunk, so the model has to pick the right one out of all of them.

In [ ]:
rng = random.Random(SEED)
chunk_ids = [c["id"] for c in chunks]
test_ids = set(rng.sample(chunk_ids, max(1, int(len(chunk_ids) * TEST_FRACTION))))

train_rows = [r for r in generated if r["chunk_id"] not in test_ids] + glossary_rows
test_rows  = [r for r in generated if r["chunk_id"] in test_ids]

corpus_ids   = chunk_ids                     # evaluation searches over all real chunks
corpus_texts = [chunk_by_id[i] for i in corpus_ids]
print(f"train pairs: {len(train_rows)} ({len(glossary_rows)} from the glossary) | "
      f"test queries: {len(test_rows)} over {len(test_ids)} held-out chunks | corpus: {len(corpus_ids)} chunks")

## 7. Load the base embedding model and measure it

In [ ]:
model = FastSentenceTransformer.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    full_finetuning = False,
)
# LoRA adapters start at zero, so the model still behaves exactly like the base model until training.
model = FastSentenceTransformer.get_peft_model(
    model,
    r = LORA_RANK,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = LORA_RANK,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    task_type = "FEATURE_EXTRACTION",
)

In [ ]:
import math

def embed(texts, is_query=False, batch_size=32):
    """🔌 HOOK — the one function your RAG pipeline needs: text -> normalized vectors."""
    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float32):
        return model.encode(texts, prompt=QUERY_PROMPT if is_query else None, batch_size=batch_size,
                            convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)

def rank_metrics(ranked_ids, gold_ids, ks=(1, 3, 5, 10)):
    """ranked_ids: list of ranked corpus-id lists; gold_ids: the correct id per query."""
    n = len(gold_ids)
    ranks = [r.index(g) + 1 if g in r else None for r, g in zip(ranked_ids, gold_ids)]
    m = {f"Recall@{k}": sum(1 for x in ranks if x and x <= k) / n for k in ks}
    m["MRR@10"]  = sum(1 / x for x in ranks if x and x <= 10) / n
    m["NDCG@10"] = sum(1 / math.log2(x + 1) for x in ranks if x and x <= 10) / n
    return m, ranks

def evaluate(rows, label):
    q = embed([r["query"] for r in rows], is_query=True)
    d = embed(corpus_texts)
    top = (q @ d.T).topk(min(10, len(corpus_ids)), dim=1).indices.cpu().tolist()
    ranked = [[corpus_ids[j] for j in row] for row in top]
    metrics, ranks = rank_metrics(ranked, [r["chunk_id"] for r in rows])
    print(f"{label:>12}: " + "  ".join(f"{k} {v:.3f}" for k, v in metrics.items()))
    return metrics, ranks

base_metrics, base_ranks = evaluate(test_rows, "base model")

## 8. Mine hard negatives

For each training question, find the passage the *base* model wrongly ranks highest. Passages scoring almost as high as the true answer are skipped, because they may actually be correct too (a "false negative").

In [ ]:
from datasets import Dataset

def mine_hard_negatives(rows, margin=0.95):
    pool_ids = [i for i in chunk_by_id if i not in test_ids]   # never use held-out passages
    pool_emb = embed([chunk_by_id[i] for i in pool_ids])
    q_emb = embed([r["query"] for r in rows], is_query=True)
    sims = (q_emb @ pool_emb.T).cpu()
    index = {cid: i for i, cid in enumerate(pool_ids)}
    out = []
    for r, s in zip(rows, sims):
        pos = index[r["chunk_id"]]
        ceiling = s[pos].item() * margin
        neg = next((j for j in s.argsort(descending=True).tolist()
                    if j != pos and s[j].item() < ceiling), None)
        if neg is None:
            neg = rng.choice([j for j in range(len(pool_ids)) if j != pos])
        out.append({"anchor": r["query"], "positive": chunk_by_id[r["chunk_id"]],
                    "negative": chunk_by_id[pool_ids[neg]]})
    return out

triplets = mine_hard_negatives(train_rows)
train_dataset = Dataset.from_list(triplets).shuffle(seed=SEED)
ex = triplets[0]
print(f"{len(triplets)} triplets\n\nanchor:   {ex['anchor']}\npositive: {ex['positive'][:150]}...\nnegative: {ex['negative'][:150]}...")

## 9. Fine-tune

`MultipleNegativesRankingLoss` pulls each question toward its passage and away from its hard negative *and* from every other passage in the batch. `prompts` makes sure the query instruction is applied to questions during training exactly as at search time.

In [ ]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
try:
    from sentence_transformers import losses
    from sentence_transformers.training_args import BatchSamplers
except ImportError:  # newer package layout
    from sentence_transformers.sentence_transformer import losses
    from sentence_transformers.base.sampler import BatchSamplers

loss = losses.MultipleNegativesRankingLoss(model)

trainer = SentenceTransformerTrainer(
    model = model,
    train_dataset = train_dataset,
    loss = loss,
    args = SentenceTransformerTrainingArguments(
        output_dir = "checkpoints",
        num_train_epochs = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        learning_rate = LEARNING_RATE,
        warmup_ratio = 0.1,
        lr_scheduler_type = "cosine",
        fp16 = not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        prompts = {"anchor": QUERY_PROMPT},          # instruction on queries only
        batch_sampler = BatchSamplers.NO_DUPLICATES,  # never put the same passage twice in a batch
        logging_steps = 5,
        save_strategy = "no",
        report_to = "none",
        seed = SEED,
    ),
)

start_mem = torch.cuda.max_memory_reserved() / 1024**3
stats = trainer.train()
print(f"\nTrained in {stats.metrics['train_runtime']/60:.1f} min; "
      f"peak GPU memory {torch.cuda.max_memory_reserved()/1024**3:.1f} GB "
      f"of {torch.cuda.get_device_properties(0).total_memory/1024**3:.0f} GB")

## 10. Measure again

In [ ]:
tuned_metrics, tuned_ranks = evaluate(test_rows, "fine-tuned")
print("\nchange:      " + "  ".join(f"{k} {tuned_metrics[k]-base_metrics[k]:+.3f}" for k in base_metrics))

def fmt(r): return str(r) if r else ">10"
moved = sorted(range(len(test_rows)), key=lambda i: (tuned_ranks[i] or 11) - (base_ranks[i] or 11))
print("\nBiggest improvements (rank of the correct passage, before -> after):")
for i in moved[:5]:
    print(f"  {fmt(base_ranks[i]):>3} -> {fmt(tuned_ranks[i]):<3} {test_rows[i]['query']}")

With a corpus this small the numbers are noisy — a handful of test questions decide each metric. Treat it as a smoke test of the pipeline; a real evaluation needs a few hundred questions, ideally some written by real users rather than the generator.

## 11. 🔌 HOOK — Try it: `search()`

This is the shape you would wire into a RAG pipeline or a vector database: embed the corpus once, embed each query with the instruction, take the top matches.

In [ ]:
corpus_emb = embed(corpus_texts)

def search(query, k=3):
    scores = (embed([query], is_query=True) @ corpus_emb.T)[0]
    top = scores.topk(k)
    return [(corpus_ids[i], round(s, 3), corpus_texts[i][:120] + "...")
            for s, i in zip(top.values.tolist(), top.indices.tolist())]

for q in ["robot stuck flashing amber and won't move",
          "what is an L-2xx error",
          "how do I add a new robot to the fleet"]:
    print(f"\n{q}")
    for cid, s, preview in search(q):
        print(f"  {s:.3f}  {cid:<22} {preview}")

## 12. Save

The LoRA adapter is small (tens of MB). The merged 16-bit model is a normal sentence-transformers model you can load anywhere — sentence-transformers, TEI, vLLM, LangChain, a vector DB ingest job.

In [ ]:
model.save_pretrained(OUTPUT_DIR)            # LoRA adapter only
model.tokenizer.save_pretrained(OUTPUT_DIR)

SAVE_MERGED = True
if SAVE_MERGED:
    model.save_pretrained_merged(OUTPUT_DIR + "_merged", tokenizer=model.tokenizer, save_method="merged_16bit")

if HF_REPO:   # needs a token: Colab sidebar -> Secrets -> HF_TOKEN
    from google.colab import userdata
    model.push_to_hub_merged(HF_REPO, tokenizer=model.tokenizer, save_method="merged_16bit", token=userdata.get("HF_TOKEN"))

# Download the adapter and the synthetic library to your computer
!zip -qr poc_outputs.zip {OUTPUT_DIR} {PAIRS_CACHE}
from google.colab import files; files.download("poc_outputs.zip")

In [ ]:
# Reload check: the merged model works with plain sentence-transformers, no Unsloth needed.
if SAVE_MERGED:
    from sentence_transformers import SentenceTransformer
    try:
        st = SentenceTransformer(OUTPUT_DIR + "_merged", device="cuda")
        q = st.encode(["robot stuck flashing amber"], prompt=QUERY_PROMPT, normalize_embeddings=True)
        d = st.encode(corpus_texts, normalize_embeddings=True)
        print("top hit:", corpus_ids[int((q @ d.T).argmax())])
    except Exception as e:
        print("Reload check failed (the saved files are still there):", e)

## 13. Scaling up to Qwen3-Embedding-8B on 2 × 96 GB

The same code carries over; these are the changes:

| Setting | Colab PoC | 2 × 96 GB box |
|---|---|---|
| `BASE_MODEL` | `unsloth/Qwen3-Embedding-0.6B` | `Qwen/Qwen3-Embedding-8B` (or `unsloth/Qwen3-Embedding-8B` if available) |
| `GENERATOR_MODEL` | Qwen2.5-3B-Instruct | a much stronger model (local 30–70B, or a hosted API) — question quality is the biggest lever |
| `QUERIES_PER_CHUNK` | 5 | 5–10, across thousands of chunks |
| `MAX_SEQ_LEN` | 384 | 512–1024 |
| `BATCH_SIZE` | 32 | 128–512 per GPU; switch to `CachedMultipleNegativesRankingLoss` for larger effective batches |
| `LORA_RANK` | 32 | 32–64, bf16 (no 4-bit needed) |

**Two GPUs:** export the cells from section 4 onward to `train.py` and run `accelerate launch --num_processes 2 train.py` (DDP; set `ddp_find_unused_parameters=False` in the training arguments). Pass `gather_across_devices=True` to the loss so each GPU also uses the other GPU's batch as negatives — available in recent sentence-transformers releases.

**Before trusting the numbers:** add 100+ real user questions with known answers to the test set, and re-embed the whole corpus with the new model before swapping it into production.